# Isolated Model 1 Colab benchmark — 2026-10-06
Free compute only; one account and runtime. 40 epochs, seed 0, frozen source e0424f99e3f668e1722e56fac4f17372cd60a62d.
12 cumulative GPU allocation hours within 72 calendar hours from first allocation, including setup, diagnostics, staging and recovery.
At most two connection attempts per UTC day; no retry loops. Keep this notebook and uploaded artifacts private.
Upload only the provided inputs-bundle.tar into My Drive/colab-benchmark-20261006/. Do not upload credentials or .env.
Use CPU for authorization and preparation. Each GPU cell releases its runtime after the phase, failure or budget expiry.
No test, training-set or new-park holdout evaluation. Sequential decoding, full precision and content verification; caching disabled throughout Colab.
Do not Run All. Execute the numbered steps individually. Results and all epoch checkpoints stay in private Drive/results/.


## 1. CPU: authorize Drive, verify uploaded inputs and record the first connection attempt

In [ ]:
from google.colab import drive
from pathlib import Path
import base64, importlib.util, shutil, time
# Run this cell on CPU. Upload inputs-bundle.tar into this PRIVATE My Drive folder first.
drive.mount('/content/drive')
FOLDER = Path('/content/drive/MyDrive/colab-benchmark-20261006')
LOCAL = Path('/content/colab-benchmark-bootstrap'); LOCAL.mkdir(exist_ok=True)
(LOCAL/'benchmark.py').write_bytes(base64.b64decode('...'))
(LOCAL/'supervisor.py').write_bytes(base64.b64decode('IiIiUnVuIGluIENvbGFiJ3Mgbm90ZWJvb2sga2VybmVsOyB0cmFpbmluZyBydW5zIG9ubHkgaW4gdGhlIHBpbm5lZCAudmVudi4KCkNQVSBwcmVwYXJhdGlvbiBtb3VudHMgRHJpdmUsIHB1Ymxpc2hlcyB0aGUgYnVuZGxlLCBhbmQgcmVjb3JkcyBvbmUgYXR0ZW1wdApiZWZvcmUgYW4gb3BlcmF0b3Igc2VsZWN0cyBhIGZyZWUgVDQuIEV2ZXJ5IGFsbG9jYXRlZCBHUFUgc2Vzc2lvbiBpcyBib3VuZGVkCmJ5IHRoZSByZW1haW5pbmcgbGVkZ2VyIGJ1ZGdldCwgaW5jbHVkaW5nIGluc3RhbGxhdGlvbiBhbmQgc3RhZ2luZy4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKaW1wb3J0IGltcG9ydGxpYi51dGlsCmltcG9ydCBqc29uCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHNodXRpbAppbXBvcnQgc3VicHJvY2VzcwppbXBvcnQgc3lzCmltcG9ydCB0aHJlYWRpbmcKaW1wb3J0IHRpbWUKCgpkZWYgbG9hZF9oYXJuZXNzKHBhdGgpOgogICAgc3BlYyA9IGltcG9ydGxpYi51dGlsLnNwZWNfZnJvbV9maWxlX2xvY2F0aW9uKCJjb2xhYl9iZW5jaG1hcmsiLCBwYXRoKQogICAgbW9kdWxlID0gaW1wb3J0bGliLnV0aWwubW9kdWxlX2Zyb21fc3BlYyhzcGVjKQogICAgc3BlYy5sb2FkZXIuZXhlY19tb2R1bGUobW9kdWxlKQogICAgcmV0dXJuIG1vZHVsZQoKCmRlZiBwcmVwYXJlKGJ1bmRsZSwgZHJpdmVfZm9sZGVyKToKICAgICIiIkV4ZWN1dGUgb24gQ1BVIGJlZm9yZSByZXF1ZXN0aW5nIGEgR1BVOyBEcml2ZSBhdXRob3JpemF0aW9uIGlzIGludGVyYWN0aXZlLiIiIgogICAgYnVuZGxlLCBkcml2ZV9mb2xkZXIgPSBQYXRoKGJ1bmRsZSksIFBhdGgoZHJpdmVfZm9sZGVyKQogICAgZnJvbSBnb29nbGUuY29sYWIgaW1wb3J0IGRyaXZlCiAgICBkcml2ZS5tb3VudCgiL2NvbnRlbnQvZHJpdmUiKQogICAgZHJpdmVfZm9sZGVyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIGhhcm5lc3MgPSBsb2FkX2hhcm5lc3MoYnVuZGxlIC8gImJlbmNobWFyay5weSIpCiAgICAjIEV4cGxpY2l0IGFsbG93bGlzdC4gTmV2ZXIgdXBsb2FkIG9wZXJhdG9yIC52ZW52LCBHb29nbGUvTW9kYWwgdG9rZW5zLCAuZW52LAogICAgIyBjaGVja291dCBHaXQgbWV0YWRhdGEsIG9yIGFueSBvdGhlciBsb2NhbCBmaWxlcy4KICAgIGxvY2sgPSBqc29uLmxvYWRzKChidW5kbGUgLyAiaW5wdXRzL2lucHV0LWxvY2suanNvbiIpLnJlYWRfdGV4dCgpKQogICAgZm9yIG5hbWUgaW4gWyJpbnB1dC1sb2NrLmpzb24iLCAqbG9ja1siZmlsZXMiXV06CiAgICAgICAgaGFybmVzcy5jb3B5X3ZlcmlmaWVkKGJ1bmRsZSAvICJpbnB1dHMiIC8gbmFtZSwgZHJpdmVfZm9sZGVyIC8gImlucHV0cyIgLyBuYW1lKQogICAgZm9yIG5hbWUgaW4gKCJiZW5jaG1hcmsucHkiLCAic3VwZXJ2aXNvci5weSIpOgogICAgICAgIGhhcm5lc3MuY29weV92ZXJpZmllZChidW5kbGUgLyBuYW1lLCBkcml2ZV9mb2xkZXIgLyBuYW1lKQogICAgbGVkZ2VyX3BhdGggPSBkcml2ZV9mb2xkZXIgLyAicmVzdWx0cy9sZWRnZXIuanNvbiIKICAgIGxlZGdlciA9IGpzb24ubG9hZHMobGVkZ2VyX3BhdGgucmVhZF90ZXh0KCkpIGlmIGxlZGdlcl9wYXRoLmV4aXN0cygpIGVsc2Uge30KICAgIGhhcm5lc3MucmVnaXN0ZXJfYXR0ZW1wdChsZWRnZXIpCiAgICBsZWRnZXJbInByZXBhcmVkX2F0dGVtcHRfYXQiXSA9IGxlZGdlclsiY29ubmVjdGlvbl9hdHRlbXB0cyJdWy0xXVsiYXQiXQogICAgaGFybmVzcy53cml0ZV9qc29uKGxlZGdlcl9wYXRoLCBsZWRnZXIpCiAgICBwcmludCgiUHJpdmF0ZSBhcnRpZmFjdHMgdmVyaWZpZWQ7IG9uZSBhdHRlbXB0IHJlY29yZGVkLiBTZWxlY3QgZnJlZSBUNCwgbm9ybWFsIFJBTS4gTm8gUHJvIG9yIHBhaWQgY3JlZGl0cy4iKQoKCmRlZiBleGVjdXRlKGRyaXZlX2ZvbGRlciwgcGhhc2UpOgogICAgZnJvbSBnb29nbGUuY29sYWIgaW1wb3J0IGRyaXZlLCBydW50aW1lCiAgICBpbXBvcnQgcHN1dGlsCiAgICBkcml2ZS5tb3VudCgiL2NvbnRlbnQvZHJpdmUiKQogICAgZm9sZGVyID0gUGF0aChkcml2ZV9mb2xkZXIpCiAgICBoYXJuZXNzID0gbG9hZF9oYXJuZXNzKGZvbGRlciAvICJiZW5jaG1hcmsucHkiKQogICAgbGVkZ2VyX3BhdGggPSBmb2xkZXIgLyAicmVzdWx0cy9sZWRnZXIuanNvbiIKICAgIGxlZGdlciA9IGpzb24ubG9hZHMobGVkZ2VyX3BhdGgucmVhZF90ZXh0KCkpCiAgICBwZXJzaXN0ZW50ID0gZm9sZGVyIC8gInJlc3VsdHMiCiAgICAjIEFsbG9jYXRpb24gaXMgY29uc2VydmF0aXZlbHkgY2hhcmdlZCBmcm9tIFZNIGJvb3QsIGluY2x1ZGluZyB0aW1lIHNwZW50CiAgICAjIHdhaXRpbmcgZm9yIERyaXZlIGF1dGhvcml6YXRpb24uIEhhcmR3YXJlIGlkZW50aXR5IGlzIGNhcHR1cmVkIGJlZm9yZSBzZXR1cC4KICAgIGJvb3RfaWQgPSBQYXRoKCIvcHJvYy9zeXMva2VybmVsL3JhbmRvbS9ib290X2lkIikucmVhZF90ZXh0KCkuc3RyaXAoKQogICAgZ3B1X2luZm8gPSBzdWJwcm9jZXNzLnJ1bihbIm52aWRpYS1zbWkiLCAiLS1xdWVyeS1ncHU9bmFtZSx1dWlkLGRyaXZlcl92ZXJzaW9uLG1lbW9yeS50b3RhbCIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICItLWZvcm1hdD1jc3Ysbm9oZWFkZXIiXSwgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwgdGV4dD1UcnVlKQogICAgaWYgZ3B1X2luZm8ucmV0dXJuY29kZToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlNlbGVjdCBhIGZyZWUgR1BVIHJ1bnRpbWUgYmVmb3JlIHRoaXMgY2VsbCIpCiAgICBpZiBhbnkoInJlbGVhc2VkX2F0IiBub3QgaW4gcyBmb3IgcyBpbiBsZWRnZXIuZ2V0KCJzZXNzaW9ucyIsIFtdKSk6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJDbG9zZSB0aGUgcHJldmlvdXMgYWxsb2NhdGlvbiBpbiB0aGUgbGVkZ2VyIGJlZm9yZSByZWNvbm5lY3RpbmciKQogICAgaWYgbGVkZ2VyLmdldCgicHJlcGFyZWRfYXR0ZW1wdF9hdCIpIGlzIE5vbmU6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJSdW4gdGhlIENQVSBwcmVwYXJhdGlvbi9hdHRlbXB0IGNlbGwgYmVmb3JlIHJlcXVlc3RpbmcgYSBHUFUiKQogICAgc2Vzc2lvbnMgPSBsZWRnZXIuc2V0ZGVmYXVsdCgic2Vzc2lvbnMiLCBbXSkKICAgIGlmIGFueShzWyJib290X2lkIl0gPT0gYm9vdF9pZCBmb3IgcyBpbiBzZXNzaW9ucyk6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJUaGlzIFZNIHdhcyBhbHJlYWR5IHVzZWQ7IHJlY292ZXJ5IHJlcXVpcmVzIGRlbGV0aW5nIHRoZSBydW50aW1lIikKICAgIGlmIHBoYXNlID09ICJjb21wbGV0aW9uIiBhbmQgbm90IGFueShzLmdldCgib3V0Y29tZSIpID09ICJhd2FpdGluZ19ydW50aW1lX2RlbGV0aW9uIgogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGZvciBzIGluIHNlc3Npb25zKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkVwb2NoLTUgZGVsZXRpb24gYm91bmRhcnkgbXVzdCBwcmVjZWRlIGNvbXBsZXRpb24iKQogICAgc2Vzc2lvbl9pZCA9IGYic2Vzc2lvbi17bGVuKHNlc3Npb25zKSsxfSIKICAgIHNlc3Npb24gPSB7ImlkIjogc2Vzc2lvbl9pZCwgImJvb3RfaWQiOiBib290X2lkLCAiYWxsb2NhdGVkX2F0IjogcHN1dGlsLmJvb3RfdGltZSgpLAogICAgICAgICAgICAgICAiYWxsb2NhdGlvbl90aW1lX2Jhc2lzIjogImNvbnNlcnZhdGl2ZSBWTSBib290IHRpbWVzdGFtcCIsICJncHVfaWRlbnRpdHkiOiBncHVfaW5mby5zdGRvdXQuc3RyaXAoKSwKICAgICAgICAgICAgICAgInBoYXNlIjogcGhhc2UsICJhdHRlbXB0X2F0IjogbGVkZ2VyLnBvcCgicHJlcGFyZWRfYXR0ZW1wdF9hdCIpfQogICAgc2Vzc2lvbnMuYXBwZW5kKHNlc3Npb24pCiAgICBoYXJuZXNzLndyaXRlX2pzb24obGVkZ2VyX3BhdGgsIGxlZGdlcikKICAgIHNlY29uZHMgPSBoYXJuZXNzLnJlbWFpbmluZyhsZWRnZXIpCiAgICBkZWFkbGluZSA9IHRpbWUudGltZSgpICsgc2Vjb25kcwogICAgaWYgc2Vjb25kcyA8PSAwOgogICAgICAgIHNlc3Npb25bImNhdXNlIl0gPSAiYnVkZ2V0IGV4aGF1c3RlZCBiZWZvcmUgc2V0dXAiCiAgICAgICAgaGFybmVzcy53cml0ZV9qc29uKGxlZGdlcl9wYXRoLCBsZWRnZXIpCiAgICAgICAgcnVudGltZS51bmFzc2lnbigpCiAgICAgICAgcmV0dXJuCiAgICBsb2NhbCA9IFBhdGgoIi9jb250ZW50L21vZGVsMS1iZW5jaG1hcmsiKQogICAgbG9jYWwubWtkaXIoZXhpc3Rfb2s9VHJ1ZSkKICAgIGxvZ19kaXIgPSBwZXJzaXN0ZW50IC8gInNlc3Npb25zIiAvIHNlc3Npb25faWQKICAgIGxvZ19kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgc3RvcF9ldmVudCA9IHRocmVhZGluZy5FdmVudCgpCiAgICBwcm9jZXNzX2hvbGRlciA9IFtdCgogICAgZGVmIGV4cGlyZSgpOgogICAgICAgIGlmIHN0b3BfZXZlbnQud2FpdChtYXgoMCwgZGVhZGxpbmUtdGltZS50aW1lKCkpKToKICAgICAgICAgICAgcmV0dXJuCiAgICAgICAgZm9yIHByb2MgaW4gcHJvY2Vzc19ob2xkZXI6CiAgICAgICAgICAgIGlmIHByb2MucG9sbCgpIGlzIE5vbmU6CiAgICAgICAgICAgICAgICBwcm9jLmtpbGwoKQogICAgICAgIHNlc3Npb25bIm91dGNvbWUiXSA9ICJidWRnZXRfZXhwaXJlZCIKICAgICAgICBzZXNzaW9uWyJyZWxlYXNlX3JlcXVlc3RlZF9hdCJdID0gdGltZS50aW1lKCkKICAgICAgICBsZWRnZXJbImNhdXNlIl0gPSAiYWdncmVnYXRlIGFsbG9jYXRpb24gb3IgNzItaG91ciBjYWxlbmRhciBidWRnZXQgZXhwaXJlZCIKICAgICAgICBoYXJuZXNzLndyaXRlX2pzb24obGVkZ2VyX3BhdGgsIGxlZGdlcikKICAgICAgICAjIE5vIHBpbmcsIHJlY29ubmVjdCwgcmV0cnksIG9yIGJhY2tncm91bmQga2VlcC1hbGl2ZS4KICAgICAgICBydW50aW1lLnVuYXNzaWduKCkKICAgIHRocmVhZGluZy5UaHJlYWQodGFyZ2V0PWV4cGlyZSwgZGFlbW9uPVRydWUpLnN0YXJ0KCkKCiAgICBkZWYgY29tbWFuZChjbWQsIGxvZyk6CiAgICAgICAgaWYgdGltZS50aW1lKCkgPj0gZGVhZGxpbmU6CiAgICAgICAgICAgIHJhaXNlIFRpbWVvdXRFcnJvcigiYWxsb2NhdGlvbiBidWRnZXQgZXhwaXJlZCIpCiAgICAgICAgd2l0aCBsb2cub3BlbigiYSIpIGFzIG91dHB1dDoKICAgICAgICAgICAgcHJvYyA9IHN1YnByb2Nlc3MuUG9wZW4oY21kLCBzdGRvdXQ9b3V0cHV0LCBzdGRlcnI9c3VicHJvY2Vzcy5TVERPVVQpCiAgICAgICAgICAgIHByb2Nlc3NfaG9sZGVyLmFwcGVuZChwcm9jKQogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICByYyA9IHByb2Mud2FpdCh0aW1lb3V0PW1heCgwLjAxLCBkZWFkbGluZS10aW1lLnRpbWUoKSkpCiAgICAgICAgICAgIGV4Y2VwdCBzdWJwcm9jZXNzLlRpbWVvdXRFeHBpcmVkOgogICAgICAgICAgICAgICAgcHJvYy5raWxsKCk7IHByb2Mud2FpdCgpCiAgICAgICAgICAgICAgICByYWlzZSBUaW1lb3V0RXJyb3IoImFsbG9jYXRpb24gYnVkZ2V0IGV4cGlyZWQiKQogICAgICAgICAgICBpZiByYzoKICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmImNvbW1hbmQgZmFpbGVkIHJjPXtyY306IHtjbWR9IikKCiAgICBvdXRjb21lID0gImluY29tcGxldGUiCiAgICB0cnk6CiAgICAgICAgc2V0dXBfc3RhcnRlZCA9IHRpbWUubW9ub3RvbmljKCkKICAgICAgICBpbnN0YWxsX2xvZyA9IGxvZ19kaXIgLyAiaW5zdGFsbC5sb2ciCiAgICAgICAgY29tbWFuZChbc3lzLmV4ZWN1dGFibGUsICItbSIsICJwaXAiLCAiaW5zdGFsbCIsICJ1diJdLCBpbnN0YWxsX2xvZykKICAgICAgICBjb21tYW5kKFsidXYiLCAicHl0aG9uIiwgImluc3RhbGwiLCAiMy4xMSJdLCBpbnN0YWxsX2xvZykKICAgICAgICBjb21tYW5kKFsidXYiLCAidmVudiIsICItLXB5dGhvbiIsICIzLjExIiwgc3RyKGxvY2FsIC8gIi52ZW52IildLCBpbnN0YWxsX2xvZykKICAgICAgICBweXRob24gPSBzdHIobG9jYWwgLyAiLnZlbnYvYmluL3B5dGhvbiIpCiAgICAgICAgY29tbWFuZChbInV2IiwgInBpcCIsICJpbnN0YWxsIiwgIi0tcHl0aG9uIiwgcHl0aG9uLAogICAgICAgICAgICAgICAgICpbZiJ7cH09PXt2fSIgZm9yIHAsdiBpbiBoYXJuZXNzLlBBQ0tBR0VTLml0ZW1zKCldXSwgaW5zdGFsbF9sb2cpCiAgICAgICAgaGFybmVzcy53cml0ZV9qc29uKGxvZ19kaXIgLyAiaW5zdGFsbGF0aW9uLmpzb24iLCB7InNlY29uZHMiOnRpbWUubW9ub3RvbmljKCktc2V0dXBfc3RhcnRlZH0pCiAgICAgICAgY29tbWFuZChbcHl0aG9uLCBzdHIoZm9sZGVyIC8gImJlbmNobWFyay5weSIpLCAicnVuIiwgIi0taW5wdXRzIiwgc3RyKGZvbGRlciAvICJpbnB1dHMiKSwKICAgICAgICAgICAgICAgICAiLS1wZXJzaXN0ZW50Iiwgc3RyKHBlcnNpc3RlbnQpLCAiLS1sb2NhbCIsIHN0cihsb2NhbCksICItLXNlc3Npb24taWQiLCBzZXNzaW9uX2lkLAogICAgICAgICAgICAgICAgICItLXBoYXNlIiwgcGhhc2VdLCBsb2dfZGlyIC8gInRyYWluaW5nLmxvZyIpCiAgICAgICAgb3V0Y29tZSA9IGpzb24ubG9hZHMoKGxvZ19kaXIgLyAib3V0Y29tZS5qc29uIikucmVhZF90ZXh0KCkpWyJzdGF0dXMiXQogICAgICAgIHNlc3Npb25bIm91dGNvbWUiXSA9IG91dGNvbWUKICAgICAgICBpZiBvdXRjb21lID09ICJjb21wbGV0ZWQiOgogICAgICAgICAgICBsZWRnZXJbImNhdXNlIl0gPSAiNDAgZXBvY2hzIGNvbXBsZXRlZCB3aXRoIHZlcmlmaWVkIHJlY292ZXJ5IgogICAgICAgIGVsaWYgb3V0Y29tZSA9PSAiYXdhaXRpbmdfcnVudGltZV9kZWxldGlvbiI6CiAgICAgICAgICAgIGxlZGdlclsiY2F1c2UiXSA9ICJlcG9jaCA1IHZlcmlmaWVkOyBhd2FpdGluZyByZWNyZWF0ZWQgcnVudGltZSIKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOgogICAgICAgIHNlc3Npb25bIm91dGNvbWUiXSA9ICJpbmNvbXBsZXRlIgogICAgICAgIHNlc3Npb25bImVycm9yIl0gPSBzdHIoZXhjKQogICAgICAgIGZhaWx1cmVfcGF0aCA9IGxvZ19kaXIgLyAiZmFpbHVyZS5qc29uIgogICAgICAgIGxlZGdlclsiY2F1c2UiXSA9IGpzb24ubG9hZHMoZmFpbHVyZV9wYXRoLnJlYWRfdGV4dCgpKVsiZXJyb3IiXSBpZiBmYWlsdXJlX3BhdGguZXhpc3RzKCkgZWxzZSBzdHIoZXhjKQogICAgICAgIHByaW50KGYiU3RvcHBlZDoge2V4Y30uIExhc3QgdmVyaWZpZWQgZXBvY2ggcmVtYWlucyBpbiBwcml2YXRlIERyaXZlLiIpCiAgICBmaW5hbGx5OgogICAgICAgIHNlc3Npb25bInJlbGVhc2VfcmVxdWVzdGVkX2F0Il0gPSB0aW1lLnRpbWUoKQogICAgICAgICMgS2VlcCB0aGUgYWxsb2NhdGlvbiBvcGVuIHVudGlsIG9wZXJhdG9yL0NMSSB2ZXJpZmllcyBhY3R1YWwgZGVsZXRpb24uCiAgICAgICAgaGFybmVzcy53cml0ZV9qc29uKGxlZGdlcl9wYXRoLCBsZWRnZXIpCiAgICAgICAgcmVmZXJlbmNlPWpzb24ubG9hZHMoKGZvbGRlciAvICJpbnB1dHMvcmVmZXJlbmNlL21vZGFsX3NlZWQwLmpzb24iKS5yZWFkX3RleHQoKSkKICAgICAgICBoYXJuZXNzLnJlcG9ydChwZXJzaXN0ZW50LCByZWZlcmVuY2UpCiAgICAgICAgc3RvcF9ldmVudC5zZXQoKQogICAgICAgIHByaW50KGYiU2Vzc2lvbiB7c2Vzc2lvbl9pZH06IHtzZXNzaW9uWydvdXRjb21lJ119LiBSZWxlYXNpbmcgcnVudGltZTsgdmVyaWZ5IGRlbGV0aW9uIGFuZCBjbG9zZSBsZWRnZXIuIikKICAgICAgICBydW50aW1lLnVuYXNzaWduKCkKCgpkZWYgY29uZmlybV9yZWxlYXNlKGZvbGRlciwgc2Vzc2lvbl9pZCwgcmVsZWFzZWRfYXQpOgogICAgIiIiQ2FsbCBvbiBDUFUgb25seSBhZnRlciB0aGUgVUkgY29uZmlybXMgZGVsZXRpb24uIFRpbWVzdGFtcCBtdXN0IGJlIG9ic2VydmVkLiIiIgogICAgZm9sZGVyID0gUGF0aChmb2xkZXIpCiAgICBoID0gbG9hZF9oYXJuZXNzKGZvbGRlciAvICJiZW5jaG1hcmsucHkiKQogICAgbGVkZ2VyX3BhdGggPSBmb2xkZXIgLyAicmVzdWx0cy9sZWRnZXIuanNvbiIKICAgIGxlZGdlciA9IGpzb24ubG9hZHMobGVkZ2VyX3BhdGgucmVhZF90ZXh0KCkpCiAgICBzZXNzaW9uID0gbmV4dChzIGZvciBzIGluIGxlZGdlclsic2Vzc2lvbnMiXSBpZiBzWyJpZCJdID09IHNlc3Npb25faWQpCiAgICBpZiAicmVsZWFzZWRfYXQiIGluIHNlc3Npb246CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigic2Vzc2lvbiByZWxlYXNlIGFscmVhZHkgcmVjb3JkZWQ7IHByZXNlcnZlIHRoZSBvcmlnaW5hbCBvYnNlcnZhdGlvbiIpCiAgICBpZiByZWxlYXNlZF9hdCA8IHNlc3Npb25bImFsbG9jYXRlZF9hdCJdOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInJlbGVhc2UgcHJlY2VkZXMgYWxsb2NhdGlvbiIpCiAgICBzZXNzaW9uWyJyZWxlYXNlZF9hdCJdID0gcmVsZWFzZWRfYXQKICAgIHNlc3Npb25bInJlbGVhc2VfdmVyaWZpZWQiXSA9IFRydWUKICAgIGgud3JpdGVfanNvbihsZWRnZXJfcGF0aCwgbGVkZ2VyKQogICAgaC5yZXBvcnQoZm9sZGVyIC8gInJlc3VsdHMiLCBqc29uLmxvYWRzKChmb2xkZXIgLyAiaW5wdXRzL3JlZmVyZW5jZS9tb2RhbF9zZWVkMC5qc29uIikucmVhZF90ZXh0KCkpKQo='))
def load(path, name):
    spec=importlib.util.spec_from_file_location(name,path)
    module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module);return module
h=load(LOCAL/'benchmark.py','harness')
s=load(LOCAL/'supervisor.py','supervisor')
h.copy_verified(FOLDER/'inputs-bundle.tar', LOCAL/'inputs-bundle.tar')
h.safe_extract(LOCAL/'inputs-bundle.tar', LOCAL/'bundle')
shutil.copyfile(LOCAL/'benchmark.py',LOCAL/'bundle/benchmark.py')
shutil.copyfile(LOCAL/'supervisor.py',LOCAL/'bundle/supervisor.py')
s.prepare(LOCAL/'bundle',FOLDER)


## 2. Select free T4, then run preflight and epochs 1–5. The full 40-epoch cosine schedule remains unchanged.

In [ ]:
# Run only after selecting FREE T4, standard RAM. Phase timing stops after epoch 5.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import importlib.util
FOLDER=Path('/content/drive/MyDrive/colab-benchmark-20261006')
spec=importlib.util.spec_from_file_location('supervisor',FOLDER/'supervisor.py')
s=importlib.util.module_from_spec(spec);spec.loader.exec_module(s)
s.execute(FOLDER, 'timing')


## 3. Verify runtime deletion in the UI. Select CPU, record release and the next attempt. If interrupted earlier, update SESSION_ID and resume the timing phase until epoch 5.

In [ ]:
# After the runtime is visibly deleted, select CPU and run this cell.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import importlib.util, time, json
FOLDER=Path('/content/drive/MyDrive/colab-benchmark-20261006')
spec=importlib.util.spec_from_file_location('supervisor',FOLDER/'supervisor.py')
s=importlib.util.module_from_spec(spec);spec.loader.exec_module(s)
SESSION_ID='session-1' # change only for subsequent allocation records
# Charge the previous allocation through the time deletion was actually observed.
s.confirm_release(FOLDER,SESSION_ID,time.time())
h=s.load_harness(FOLDER/'benchmark.py')
ledger_path=FOLDER/'results/ledger.json';ledger=json.loads(ledger_path.read_text())
h.register_attempt(ledger)
ledger['prepared_attempt_at']=ledger['connection_attempts'][-1]['at']
h.write_json(ledger_path,ledger)
print('Next attempt recorded; remaining seconds:',h.remaining(ledger))


## 4. Select free T4 again; resume epoch 6 toward epoch 40. The harness checks all restored optimizer/model/RNG/shuffle state and the original cosine learning rate.

In [ ]:
# Run only after selecting FREE T4, standard RAM. Phase timing stops after epoch 5.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import importlib.util
FOLDER=Path('/content/drive/MyDrive/colab-benchmark-20261006')
spec=importlib.util.spec_from_file_location('supervisor',FOLDER/'supervisor.py')
s=importlib.util.module_from_spec(spec);spec.loader.exec_module(s)
s.execute(FOLDER, 'completion')


## 5. Verify final runtime deletion. On CPU, close the last session and generate the report. Do not record another attempt if completed or expired.

In [ ]:
# After the runtime is visibly deleted, select CPU and run this cell.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import importlib.util, time, json
FOLDER=Path('/content/drive/MyDrive/colab-benchmark-20261006')
spec=importlib.util.spec_from_file_location('supervisor',FOLDER/'supervisor.py')
s=importlib.util.module_from_spec(spec);spec.loader.exec_module(s)
SESSION_ID='session-2' # change only for subsequent allocation records
# Charge the previous allocation through the time deletion was actually observed.
s.confirm_release(FOLDER,SESSION_ID,time.time())


Record active operator minutes, failed connection attempts, interruptions and repeated work in results/ledger.json. Unknown effort or interrupted-allocation duration must stay explicitly unknown/conservative. Compare Modal L4 cached reference with Colab uncached execution; no parallel or monthly-capacity conclusions.